# 01 - Reproduce the IDIAP text teacher

Recreates the **exact external text-teacher KD targets** from the pinned public IDIAP checkpoint. This is frozen-checkpoint inference, not random re-training, because that is how the v3 teacher was defined.

**Selection policy:** teacher epoch 0 is the original published IDIAP checkpoint. Any future local fine-tuning must beat epoch 0 under the fixed DEV criterion before it may replace the original teacher.

In [ ]:
from google.colab import drive, userdata
drive.mount('/content/drive')

from pathlib import Path
import os, sys, json, shutil, subprocess, base64

DAIC = Path('/content/drive/MyDrive/DAIC_WOZ')
EXP_ROOT = DAIC / 'experiments' / 'exact_v3_reproduction'
ACTIVE = EXP_ROOT / 'ACTIVE_RUN.txt'
assert ACTIVE.is_file(), 'Run notebooks/00_start_new_run.ipynb first.'
RUN = EXP_ROOT / ACTIVE.read_text().strip()
assert RUN.is_dir(), RUN

REPO = 'engrkhubabahmad/reliability-aware-depression-kd'
CORE_COMMIT = 'd6ed3a8b54ec9cfa7c706cb441671c6e8b07096d'
CODE = Path('/content/reliability-aware-depression-kd-exact-v3')

token = userdata.get('GITHUB_TOKEN')
assert token, 'Add GITHUB_TOKEN to Colab Secrets.'
auth = base64.b64encode(f'x-access-token:{token}'.encode()).decode()
env = os.environ.copy()
env['GIT_CONFIG_COUNT']='1'
env['GIT_CONFIG_KEY_0']='http.https://github.com/.extraheader'
env['GIT_CONFIG_VALUE_0']=f'AUTHORIZATION: basic {auth}'

if CODE.exists(): shutil.rmtree(CODE)
CODE.mkdir(parents=True)
subprocess.run(['git','-C',str(CODE),'init'],check=True,env=env)
subprocess.run(['git','-C',str(CODE),'remote','add','origin',f'https://github.com/{REPO}.git'],check=True,env=env)
subprocess.run(['git','-C',str(CODE),'fetch','--depth','1','origin',CORE_COMMIT],check=True,env=env)
subprocess.run(['git','-C',str(CODE),'checkout','--detach','FETCH_HEAD'],check=True,env=env)
head=subprocess.check_output(['git','-C',str(CODE),'rev-parse','HEAD'],text=True).strip()
assert head==CORE_COMMIT,(head,CORE_COMMIT)
subprocess.run([sys.executable,'-m','pip','install','-q','-r',str(CODE/'requirements.txt')],check=True)
print('RUN:',RUN)
print('Pinned core:',head)


In [ ]:
IDIAP_REPO='https://github.com/idiap/bias_in_daic-woz.git'
IDIAP_COMMIT='de8f477619ce61a2a65640a0735aa8dfe659ac59'
SRC=Path('/content/idiap-bias_in_daic-woz')
if SRC.exists(): shutil.rmtree(SRC)
subprocess.run(['git','clone',IDIAP_REPO,str(SRC)],check=True)
subprocess.run(['git','-C',str(SRC),'checkout','--detach',IDIAP_COMMIT],check=True)
resolved=subprocess.check_output(['git','-C',str(SRC),'rev-parse','HEAD'],text=True).strip()
assert resolved==IDIAP_COMMIT,(resolved,IDIAP_COMMIT)
print('Pinned IDIAP source:',resolved)

In [ ]:
OUT=RUN/'01_text_teacher'
OUT.mkdir(parents=True,exist_ok=False)
cmd=[sys.executable,'-m','scripts.teachers.idiap_text.export_targets',
     '--daic-root',str(DAIC),'--source-root',str(SRC),'--output',str(OUT)]
subprocess.run(cmd,cwd=CODE,check=True)

audit=json.loads((OUT/'text_target_audit.json').read_text())
m=audit['dev34']
print(json.dumps(m,indent=2))
assert audit['train_participants']==107 and audit['dev_participants']==34
assert audit['test_opened'] is False
assert round(float(m['macro_f1']),10)==round(0.8418604651162791,10)

# Teacher epoch 0 means the ORIGINAL published checkpoint.
orig_model=SRC/'model/Participant/model_inductgcn[250].pkl'
orig_vectorizer=SRC/'model/Participant/vtzer_inductgcn[250].pkl'
assert orig_model.is_file() and orig_vectorizer.is_file()

shutil.copy2(orig_model,OUT/'epoch0_original_teacher.pkl')
shutil.copy2(orig_vectorizer,OUT/'epoch0_original_vectorizer.pkl')
shutil.copy2(orig_model,OUT/'selected_teacher.pkl')
shutil.copy2(orig_vectorizer,OUT/'selected_vectorizer.pkl')
shutil.copy2(OUT/'train_text_kd_targets.csv',OUT/'selected_train_kd_targets.csv')

selection={
    'teacher':'IDIAP participant InducT-GCN top250',
    'epoch0_definition':'original published checkpoint before any local fine-tuning',
    'epoch0_source_commit':IDIAP_COMMIT,
    'epoch0_dev34':m,
    'selection_rule':'start with epoch 0; replace only on strict lexicographic DEV improvement: macro-F1, depressed-F1, AUROC',
    'fine_tuning_attempted':False,
    'selected_epoch':0,
    'selected_source':'original published checkpoint',
    'selected_targets':'selected_train_kd_targets.csv',
    'test_opened':False,
}
(OUT/'teacher_selection.json').write_text(json.dumps(selection,indent=2)+'\n')

print('PASS: IDIAP text teacher DEV-34 macro-F1 =',m['macro_f1'])
print('SELECTED TEACHER: epoch 0 original published IDIAP checkpoint')